In [541]:
%pip install mrmr-selection

  Using cached mrmr_selection-0.2.8-py3-none-any.whl.metadata (6.6 kB)
  Using cached category_encoders-2.11.2-py3-none-any.whl.metadata (8.5 kB)
  Using cached polars-2.1.0-py3-none-any.whl.metadata (11 kB)
  Using cached polars_runtime_32-2.1.0-cp311-abi3-win_amd64.whl.metadata (1.5 kB)
Using cached mrmr_selection-0.2.8-py3-none-any.whl (15 kB)
Using cached polars-2.1.0-py3-none-any.whl (911 kB)
Using cached polars_runtime_32-2.1.0-cp311-abi3-win_amd64.whl (53.9 MB)
Using cached category_encoders-2.11.2-py3-none-any.whl (108 kB)

   ---------------------------------------- 0/4 [polars-runtime-32]
   ---------------------------------------- 0/4 [polars-runtime-32]
   ---------------------------------------- 0/4 [polars-runtime-32]
   ---------------------------------------- 0/4 [polars-runtime-32]
   ---------------------------------------- 0/4 [polars-runtime-32]
   ---------- ----------------------------- 1/4 [polars]
   ---------- ----------------------------- 1/4 [polars]
   -----

In [544]:
# Import neccessary libraries
import itertools

import pandas as pd
import numpy as np

# For graph plot
import matplotlib.pyplot as plt
from matplotlib import cm
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import mutual_info_regression
from mrmr import mrmr_regression

# For the DLNM model
from statsmodels.gam.gam_cross_validation.cross_validators import KFold
import statsmodels.api as sm
from statsmodels.gam.smooth_basis import UnivariateBSplines, UnivariateCubicCyclicSplines
from statsmodels.gam.api import GLMGam, BSplines, CyclicCubicSplines
from statsmodels.gam.gam_cross_validation.gam_cross_validation import MultivariateGAMCVPath
from statsmodels.genmod.families.family import Tweedie
from statsmodels.genmod.families.links import log
from statsmodels.gam.smooth_basis import GenericSmoothers
from pydlnm.basis import CrossBasis
from pydlnm.prediction import crosspred
from pydlnm.utils import logknots
from dlnm import crossbasis, plot_crosspred, cb_pen


In [525]:
def crossbasis_construct(dataframe, col_lst, max_lag, var_degree, var_df, lag_degree, lag_df):
    cb_list = []
    for col in col_lst:
        knots_var = np.quantile(dataframe[col].dropna(), [0.1, 0.75, 0.9])
        lag_knots = logknots([0, max_lag], nk=lag_df)

        tmp_cb = CrossBasis(dataframe[col].values, lag=max_lag, 
                                argvar={"fun": "bs", "degree": var_degree, "df": var_df}, 
                                arglag={"fun": "bs", "degree": lag_degree, "df": lag_df})
        cb_list.append(tmp_cb)

    X_exposure = np.concat(cb_list, axis=1)

    model_data = np.concat([dataframe.drop(columns=["From"]).to_numpy(), X_exposure], axis=1)
    model_data = pd.DataFrame(model_data)
    exp_name_list = [f"{col}_basis_{i}" for col in col_lst for i in range(tmp_cb.shape[1])]
    model_data.columns = [n for n in dataframe.columns if n!="From"] + exp_name_list
    model_data[exp_name_list] = model_data[exp_name_list].fillna(0)
    exp_names = " + ".join(exp_name_list)

    return model_data, exp_names, cb_list

In [526]:
def smoothers_construct(dataframe, bs_dict, cs_dict):
    bs = UnivariateBSplines(dataframe[bs_dict["name"]], df=bs_dict["df"], degree=bs_dict["degree"])
    cs = UnivariateCubicCyclicSplines(dataframe[cs_dict["name"]], df=cs_dict["df"])
    sm_weights = [bs, cs]
    
    X_smooth = dataframe[[bs_dict["name"]]+[cs_dict["name"]]]
    smoothers = GenericSmoothers(X_smooth, sm_weights)
    
    return smoothers

In [527]:
def NB_GAM_model(dataframe, tar_var, exp_names, smoothers, a): 
    formula = f"{tar_var} ~ {exp_names}"

    NegativeBinomial_gam_model = GLMGam.from_formula(formula, data=dataframe, smoother=smoothers, alpha=a, family=sm.families.NegativeBinomial(alpha=0.5))
    result = NegativeBinomial_gam_model.fit()
    # result = NegativeBinomial_gam_model.fit_regularized(method="l1_slsqp", alpha=1.0)
        # no penalization for Negative Binomial

    return NegativeBinomial_gam_model, result

In [528]:
def Tweddie_GAM_model(dataframe, tar_var, exp_names, smoothers, a):
    formula = f"{tar_var} ~ {exp_names}"
    Tweddie_gam_model = GLMGam.from_formula(formula, data=dataframe, smoother=smoothers, alpha=a, family=Tweedie(link=log(), var_power=1.5))
    result = Tweddie_gam_model.fit()

    return Tweddie_gam_model, result

In [529]:
def pred_cal(result, dataframe, cb_lst, mete_lst, i):
    n_cb_cols = cb_lst[i].shape[1]
    start_idx = 0
    end_idx = start_idx+n_cb_cols
    cb_coef = result.params.values[start_idx:end_idx]
    cb_vcov = result.cov_params().iloc[start_idx:end_idx, start_idx:end_idx]
    at = np.linspace(dataframe[mete_lst[i]].min(), dataframe[mete_lst[i]].max(), 100)
    cen = dataframe[mete_lst[i]].median()
    pred = crosspred(basis=cb_lst[i], coef=cb_coef, vcov=cb_vcov, model_link="log", at=at, cen=cen)
    
    print(f"{mete_lst[i]} RR range: {pred.allRRfit.min():.3f} to {pred.allRRfit.max():.3f}")

    return pred

In [530]:
def graph_3Dplot(pred, exp):
    fig = plot_crosspred(pred, ptype="3d")

    ax = fig.gca()
    ax.set_title("3D Exposure-Lag-Response Association Surface", fontsize=14, fontweight='bold', pad=15)
    ax.set_xlabel(exp, fontsize=12, labelpad=10)
    ax.set_ylabel("Lag (Weeks)", fontsize=12, labelpad=10)
    ax.set_zlabel("Residual Risk (RR)", labelpad=10)
    ax.set_yticks(np.linspace(0, 4, pred.matRRfit.shape[1]))

    plt.tight_layout()
    plt.show()

In [531]:
def graph_expresponseplot(pred, exp):
    fig = plot_crosspred(pred, ptype="overall")
    cen = df[exp].median()

    ax = fig.gca()

    ax.axvline(x=cen, color='black', linestyle=':', linewidth=1.2, label=f'Centering Temp ({cen:.1f}°C)')
    ax.set_title('Overall Cumulative Exposure-Response Association', fontsize=14, pad=15)
    ax.set_xlabel(exp, fontsize=12)
    ax.legend(loc="best", frameon=True)

    plt.show()

In [532]:
def graph_lagresponseplot(pred, target_var):
    fig = plot_crosspred(
        pred, 
        ptype="slices", 
        var=target_var
    )

    ax = fig.gca()

    ax.set_title(f"Lag-Response Curve at {target_var}", fontsize=14, fontweight='bold')
    ax.set_xlabel("Lag (Weeks)", fontsize=12)
    ax.set_ylabel("Relative Risk (RR)", fontsize=12)
    ax.axhline(1, color='gray', linestyle='--', alpha=0.7) # Add baseline at RR=1
    
    plt.show()

In [533]:
def graph_plot(pred, exp):
    graph_3Dplot(pred, exp)
    graph_expresponseplot(pred, exp)
    graph_lagresponseplot(pred, 15)

In [534]:
def alpha_byAIC(dataframe, tar_var, exp_names, smoothers, alpha):
    best_a = 0
    best_aic = np.inf

    for a1, a2 in itertools.product(alpha, alpha):
        model, result = NB_GAM_model(dataframe, tar_var, exp_names, smoothers, [a1, a2])
        print(f"alpha {[a1, a2]} with AIC value {result.aic}")
        if result.aic < best_aic:
            best_aic = result.aic
            best_a = [a1, a2]

    return best_a

In [535]:
def alpha_byGCV(dataframe, tar_var, exp_names, smoothers):
    model, result = NB_GAM_model(dataframe, tar_var, exp_names, smoothers, [100, 500])
    best_a = model.select_penweight(criterion="gcv")
    return best_a[0]

In [546]:
def feature_scaling():
    df[mete_col] = StandardScaler().fit_transform(df[mete_col])
    df["Year"] = df["Year"] - df["Year"].mean()
  

In [ ]:
def feature_selection_MI(dataframe, exp_col, outcome_col):
    X = dataframe[exp_col]
    Y = dataframe[outcome_col]

            # Calculate mutual information for regression
    mi_scores = mutual_info_regression(X, Y)

            # Display scores
    mi_df = pd.DataFrame({'Feature': [col for col in X], 'MI_Score': mi_scores})
    selected_features = mi_df.loc[mi_df["MI_Score"]>=0, "Feature"].to_list()

    return mi_df.sort_values(by='MI_Score', ascending=False), selected_features

In [564]:
def feature_selection_MRMR(dataframe, exp_col, outcome_col):
    selected_features = mrmr_regression(X=dataframe[exp_col], y=dataframe[outcome_col], K=5)
    print("Selected Features:", selected_features)
            
    return selected_features 

---

In [579]:
# Import data and basic settings
df = pd.read_csv("data.csv")

df.columns = ["Year", "Week", "From", "temprange", "temp", "RH", "SR", "AP", "DPT", "WS", "UV", "H1", "H3", "B", "AandB"]

mete_col = ["temprange", "temp", "RH", "SR", "AP", "DPT", "WS", "UV"]
flu_col = ["H1", "H3", "B", "AandB"]


In [ ]:
# Feature scaling and selection
np.random.seed(42)

feature_scaling()
feature_selection_MI(df, mete_col, "H1")
mete_col = feature_selection_MRMR(df, mete_col, "H1")

100%|██████████| 5/5 [00:04<00:00,  1.16it/s]

Selected Features: ['UV', 'RH', 'WS', 'temp', 'SR']


In [582]:
model_data, exp_names, cb_list = crossbasis_construct(df, mete_col, 4, 2, 5, 3, 6)
smoothers = smoothers_construct(df, {"name": "Year", "degree": 2, "df": 3}, {"name": "Week", "df": 4})

In [584]:
alpha_choice = [0.01, 0.1, 1.0, 10.0, 50.0, 100.0]
best_a_aic = alpha_byAIC(df, "H1", " + ".join(mete_col), smoothers, alpha_choice)

print(f"alpha {best_a_aic} is the best tested with AIC")

alpha [0.01, 0.01] with AIC value 14661.647525055314
alpha [0.01, 0.1] with AIC value 14661.647469220912
alpha [0.01, 1.0] with AIC value 14661.646915997328
alpha [0.01, 10.0] with AIC value 14661.641805078794
alpha [0.01, 50.0] with AIC value 14661.628359643462
alpha [0.01, 100.0] with AIC value 14661.632072920896
alpha [0.1, 0.01] with AIC value 14661.647517549156
alpha [0.1, 0.1] with AIC value 14661.647461714909
alpha [0.1, 1.0] with AIC value 14661.646908492607
alpha [0.1, 10.0] with AIC value 14661.641797589047
alpha [0.1, 50.0] with AIC value 14661.628352219483
alpha [0.1, 100.0] with AIC value 14661.632065576778
alpha [1.0, 0.01] with AIC value 14661.647442502293
alpha [1.0, 0.1] with AIC value 14661.647387280547
alpha [1.0, 1.0] with AIC value 14661.646833460007
alpha [1.0, 10.0] with AIC value 14661.641722706176
alpha [1.0, 50.0] with AIC value 14661.628277995582
alpha [1.0, 100.0] with AIC value 14661.63199215007
alpha [10.0, 0.01] with AIC value 14661.646694451503
alpha [10

In [590]:
model, result = NB_GAM_model(df, "H1", " + ".join(mete_col), smoothers, best_a_aic)
print(result.summary())


                 Generalized Linear Model Regression Results                  
Dep. Variable:                     H1   No. Observations:                 1253
Model:                         GLMGam   Df Residuals:                  1242.12
Model Family:        NegativeBinomial   Df Model:                         9.88
Link Function:                    Log   Scale:                          1.0000
Method:                         PIRLS   Log-Likelihood:                -7319.9
Date:                Mon, 09 Nov 2026   Deviance:                       8482.9
Time:                        09:04:08   Pearson chi2:                 1.07e+04
No. Iterations:                    30   Pseudo R-squ. (CS):             0.9100
Covariance Type:            nonrobust                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      1.3208      0.056     23.399      0.0

In [587]:
best_a_gcv = alpha_byGCV(df, "H1", " + ".join(mete_col), smoothers)

print(f"alpha {best_a_gcv} is the best tested with GCV")


alpha [1.40878725e+43 6.69160853e+04] is the best tested with GCV


In [589]:
model, result = NB_GAM_model(df, "H1", " + ".join(mete_col), smoothers, best_a_gcv)
print(result.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:                     H1   No. Observations:                 1253
Model:                         GLMGam   Df Residuals:                     1253
Model Family:        NegativeBinomial   Df Model:                           -1
Link Function:                    Log   Scale:                          1.0000
Method:                         PIRLS   Log-Likelihood:                -94235.
Date:                Mon, 09 Nov 2026   Deviance:                   1.8231e+05
Time:                        09:04:01   Pearson chi2:                 3.38e+07
No. Iterations:                     3   Pseudo R-squ. (CS):         -1.600e+59
Covariance Type:            nonrobust                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept   9.979e-35   5.34e-29   1.87e-06      1.0

In [ ]:
pred = pred_cal(result, df, cb_list, mete_col, 0)
graph_plot(pred, mete_col[0])

ValueError: Variable basis (5) doesn't match coefficients (12)